# Parkinson AI — Colab GPU pipeline (compact checkpoints)
## WideResNet50-2 + Focal Loss → DeepRBF → combined inference

This version keeps intermediate WideResNet checkpoints resumable, but stores each final completed fold without the large optimizer/RNG state. Model weights and provenance metadata are preserved. It is designed to make the 25-fold workflow practical with a 15 GB Google Drive quota.

## 0. Connect to Colab, mount the shared Drive, and prepare the code/data

### VS Code connection

1. Install the official **Google Colab** extension in VS Code.
2. Open this `.ipynb` file.
3. Click **Select Kernel** → **Colab** → choose a Colab runtime.
4. Sign in to the Google account that has access to the team's `ParkinsonResearch` Drive folder.
5. Select a runtime with a **GPU**.

The first code cell checks CUDA and stops if the runtime is not a Colab GPU.

### Why the code comes from GitHub

The previous `parkinson_training_code_v2.zip` was made for the baseline notebook. To avoid silently using an older code snapshot, this notebook checks out the current DeepRBF-only source at a fixed Git commit. The shared dataset remains the exact versioned ZIP already used by the team.

The dataset is copied from Drive to Colab's temporary disk, checksum-verified, safely extracted, and every packaged file is verified. Model checkpoints and reports are saved back to Drive so they survive a Colab runtime reset.

In [6]:

from pathlib import Path
import hashlib
import json
import shutil
import zipfile

# ==========================================
# 1. Dataset configuration
# ==========================================

DATASET_SHA256 = (
    "85efa52f80adbd357537842146ae107e3cd6b9817eb"
    "7ec95beba2644d813f1c6"
)

DATA_ROOT = Path("/content/ParkinsonData")
DATASET_DIR = DATA_ROOT / "datasets"
MANIFEST = DATASET_DIR / "annotations" / "fold_manifest.csv"

SHORTCUT_ROOT = Path(
    "/content/drive/.shortcut-targets-by-id"
)

# ==========================================
# 2. Find shared Google Drive project
# ==========================================

assert SHORTCUT_ROOT.is_dir(), "Google Drive is not mounted"

projects = [
    p / "ParkinsonResearch"
    for p in SHORTCUT_ROOT.iterdir()
    if (p / "ParkinsonResearch" / "datasets" /
        "parkinson_dataset_v1.zip").is_file()
]

if len(projects) != 1:
    raise RuntimeError(
        f"Expected one ParkinsonResearch folder, found {len(projects)}"
    )

DRIVE_DIR = projects[0]

SOURCE_ZIP = (
    DRIVE_DIR / "datasets" / "parkinson_dataset_v1.zip"
)

RUNS_DIR = DRIVE_DIR / "runs"

print("Project:", DRIVE_DIR)
print("Dataset ZIP:", SOURCE_ZIP)

# ==========================================
# 3. SHA256 helper
# ==========================================

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


# ==========================================
# 4. Secure dataset extraction
# ==========================================

def extract_checked(source, destination, expected_sha256):

    if not source.is_file():
        raise FileNotFoundError(source)

    local_zip = Path("/content/parkinson_dataset_v1.zip")

    print("\nCopying dataset to Colab local storage...")
    shutil.copy2(source, local_zip)

    actual_sha256 = sha256_file(local_zip)

    print("Expected SHA256:", expected_sha256)
    print("Actual SHA256:  ", actual_sha256)

    if actual_sha256 != expected_sha256:
        raise ValueError(
            "Dataset SHA256 mismatch. Extraction stopped."
        )

    print("\nDataset checksum verified.")

    destination.mkdir(parents=True, exist_ok=True)

    destination_resolved = destination.resolve()

    with zipfile.ZipFile(local_zip, "r") as archive:

        for info in archive.infolist():

            target = (
                destination / info.filename
            ).resolve()

            if not target.is_relative_to(destination_resolved):
                raise ValueError(
                    f"Unsafe archive path: {info.filename}"
                )

        archive.extractall(destination)

    print("Dataset extraction completed.")


# ==========================================
# 5. Extract dataset
# ==========================================

metadata_path = DATA_ROOT / "dataset_package.json"

if metadata_path.is_file() and MANIFEST.is_file():
    print("\nExisting extracted dataset detected.")
else:
    extract_checked(
        SOURCE_ZIP,
        DATA_ROOT,
        DATASET_SHA256
    )


# ==========================================
# 6. Verify every dataset file
# ==========================================

if not metadata_path.is_file():
    raise FileNotFoundError("dataset_package.json missing")

metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

if metadata.get("dataset_version") != "parkinson_dataset_v1":
    raise ValueError("Unexpected dataset version")

for name, info in metadata["files"].items():

    target = (DATA_ROOT / name).resolve()

    if not target.is_relative_to(DATA_ROOT.resolve()):
        raise ValueError(f"Unsafe dataset metadata path: {name}")

    if not target.is_file():
        raise FileNotFoundError(
            f"Dataset file missing: {target}"
        )

    actual = sha256_file(target)

    if actual != info["sha256"]:
        raise ValueError(
            f"Dataset file checksum mismatch: {name}"
        )

print("All dataset files verified.")

# ==========================================
# 7. Final paths
# ==========================================

if not MANIFEST.is_file():
    raise FileNotFoundError(
        f"Fold manifest missing: {MANIFEST}"
    )

RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 50)
print("DATASET READY")
print("=" * 50)

print("Version:", metadata["dataset_version"])
print("Images:", metadata["images"])
print("Traces:", metadata["traces"])
print("Dataset:", DATASET_DIR)
print("Manifest:", MANIFEST)
print("Runs:", RUNS_DIR)
print("SHA256: VERIFIED")


Project: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch
Dataset ZIP: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/datasets/parkinson_dataset_v1.zip

Copying dataset to Colab local storage...
Expected SHA256: 85efa52f80adbd357537842146ae107e3cd6b9817eb7ec95beba2644d813f1c6
Actual SHA256:   85efa52f80adbd357537842146ae107e3cd6b9817eb7ec95beba2644d813f1c6

Dataset checksum verified.
Dataset extraction completed.
All dataset files verified.

DATASET READY
Version: parkinson_dataset_v1
Images: 147
Traces: 147
Dataset: /content/ParkinsonData/datasets
Manifest: /content/ParkinsonData/datasets/annotations/fold_manifest.csv
Runs: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs
SHA256: VERIFIED


In [5]:

from pathlib import Path

# 1. Check mounted Google Drive
DRIVE_ROOT = Path("/content/drive")
SHORTCUT_ROOT = DRIVE_ROOT / ".shortcut-targets-by-id"

print("===== GOOGLE DRIVE =====")
print("Drive mounted:", DRIVE_ROOT.exists())
print("Shortcut directory:", SHORTCUT_ROOT.exists())

if not SHORTCUT_ROOT.exists():
    raise RuntimeError(
        "Google Drive shortcut directory is unavailable. "
        "Reconnect the Colab runtime."
    )

# 2. Find the actual shared project folder
matches = []

for shortcut in SHORTCUT_ROOT.iterdir():
    project = shortcut / "ParkinsonResearch"
    archive = (
        project / "datasets" / "parkinson_dataset_v1.zip"
    )

    if archive.is_file():
        matches.append(project)

print("\n===== PROJECT SEARCH =====")
print("Matches:", len(matches))

for item in matches:
    print("FOUND:", item)

if len(matches) != 1:
    raise RuntimeError(
        f"Expected exactly one project folder, found {len(matches)}"
    )

# 3. Set all paths from the discovered folder
DRIVE_DIR = matches[0]

SOURCE_ZIP = (
    DRIVE_DIR / "datasets" / "parkinson_dataset_v1.zip"
)

RUNS_DIR = DRIVE_DIR / "runs"

DATA_ROOT = Path("/content/ParkinsonData")
DATASET_DIR = DATA_ROOT / "datasets"
MANIFEST = (
    DATASET_DIR / "annotations" / "fold_manifest.csv"
)

print("\n===== FINAL PATHS =====")
print("DRIVE_DIR:", DRIVE_DIR)
print("SOURCE_ZIP:", SOURCE_ZIP)
print("RUNS_DIR:", RUNS_DIR)

print("\n===== VERIFICATION =====")
print("Drive:", DRIVE_DIR.exists())
print("Dataset ZIP:", SOURCE_ZIP.is_file())
print("Runs:", RUNS_DIR.is_dir())

# 4. Check whether extraction function and checksum exist
print("\n===== NOTEBOOK FUNCTIONS =====")
print(
    "extract_checked defined:",
    callable(globals().get("extract_checked"))
)
print(
    "DATASET_SHA256 defined:",
    bool(globals().get("DATASET_SHA256"))
)


===== GOOGLE DRIVE =====
Drive mounted: True
Shortcut directory: True

===== PROJECT SEARCH =====
Matches: 1
FOUND: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch

===== FINAL PATHS =====
DRIVE_DIR: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch
SOURCE_ZIP: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/datasets/parkinson_dataset_v1.zip
RUNS_DIR: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs

===== VERIFICATION =====
Drive: True
Dataset ZIP: True
Runs: True

===== NOTEBOOK FUNCTIONS =====
extract_checked defined: False
DATASET_SHA256 defined: False


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile

# This notebook must run on a managed Google Colab runtime, not a local kernel.
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "This notebook must use a Google Colab kernel. In VS Code choose "
        "Select Kernel → Colab, sign in, and connect to a GPU runtime."
    ) from exc

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is attached to this Colab runtime. In the Colab kernel/runtime "
        "selector, choose a GPU runtime, reconnect, and rerun this cell. "
        "Training has NOT started."
    )

# Shared paths used by the team's other Colab notebooks.
drive.mount("/content/drive")
from pathlib import Path

print("Drive:", DRIVE_DIR.exists())
print("Dataset ZIP:", SOURCE_ZIP.exists())
print("Runs:", RUNS_DIR.exists())

assert DRIVE_DIR.exists(), "Drive folder not found"
assert SOURCE_ZIP.is_file(), "Dataset ZIP not found"

print("\nDataset archive found:")
print(SOURCE_ZIP)

# Extract using the existing SHA256 verification function
extract_checked(
    SOURCE_ZIP,
    DATA_ROOT,
    DATASET_SHA256
)

# Verify extracted dataset
assert MANIFEST.is_file(), (
    f"Fold manifest missing: {MANIFEST}"
)

assert (DATA_ROOT / "dataset_package.json").is_file()

print("\nDataset extraction successful!")
print("Dataset:", DATASET_DIR)
print("Manifest:", MANIFEST)
SOURCE_ZIP = DRIVE_DIR / "datasets/parkinson_dataset_v1.zip"
DATA_ROOT = Path("/content/ParkinsonData")
DATASET_DIR = DATA_ROOT / "datasets"
MANIFEST = DATASET_DIR / "annotations/fold_manifest.csv"
RUNS_DIR = DRIVE_DIR / "runs"

DATASET_SHA256 = "85efa52f80adbd357537842146ae107e3cd6b9817eb7ec95beba2644d813f1c6"
REPOSITORY_URL = "https://github.com/Aihan-2005/ParkinsonAiResearch.git"
CODE_COMMIT = "a1d1bfcb3e0234c24a3c9c0c81ece6ace26ab835"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def extract_checked(source: Path, destination: Path, expected_sha256: str) -> None:
    if not source.is_file():
        raise FileNotFoundError(
            f"Required Drive archive is missing: {source}. "
            "Check that the correct Google account is connected and the shared "
            "ParkinsonResearch folder is available."
        )
    local_zip = Path("/content") / source.name
    shutil.copy2(source, local_zip)
    actual_sha256 = sha256_file(local_zip)
    if actual_sha256 != expected_sha256:
        raise ValueError(
            f"SHA-256 mismatch for {source.name}. "
            f"Expected {expected_sha256}; received {actual_sha256}."
        )

    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(local_zip) as archive:
        root = destination.resolve()
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f"Unsafe ZIP path: {member.filename}")
        archive.extractall(destination)


# 0) Load the shared dataset using the exact same version/checksum as the
# team's 00_shared_dataset_colab.ipynb and 01_wideresnet50_2.ipynb.

metadata_path = DATA_ROOT / "dataset_package.json"

# Reuse the already extracted and verified dataset.
if metadata_path.is_file() and MANIFEST.is_file():
    print("Existing extracted dataset detected.")
    print("Dataset:", DATA_ROOT)
    print("Manifest:", MANIFEST)

else:
    if not SOURCE_ZIP.is_file():
        raise FileNotFoundError(
            f"Dataset archive not found: {SOURCE_ZIP}\n"
            "The extracted dataset is also unavailable.\n"
            "Check the shared Drive folder and archive location."
        )

    print("Extracting verified dataset...")
    extract_checked(
        SOURCE_ZIP,
        DATA_ROOT,
        DATASET_SHA256
    )

metadata_path = DATA_ROOT / "dataset_package.json"

metadata_path = DATA_ROOT / "dataset_package.json"
if not metadata_path.is_file():
    raise FileNotFoundError(f"Dataset package metadata is missing: {metadata_path}")
dataset_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
for relative_name, info in dataset_metadata["files"].items():
    target = (DATA_ROOT / relative_name).resolve()
    if not target.is_relative_to(DATA_ROOT.resolve()):
        raise ValueError(f"Unsafe dataset metadata path: {relative_name}")
    if not target.is_file() or sha256_file(target) != info["sha256"]:
        raise ValueError(f"Dataset file failed integrity verification: {relative_name}")

if not MANIFEST.is_file():
    raise FileNotFoundError(f"Shared fold manifest is missing: {MANIFEST}")

# 1) Fetch the exact refactored source commit from GitHub. This commit contains
# the DeepRBF-only implementation and no active CentreRBF model path.
PROJECT_DIR = Path("/content/ParkinsonAiResearch")
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
subprocess.run(
    ["git", "clone", "--quiet", REPOSITORY_URL, str(PROJECT_DIR)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "checkout", "--quiet", "--detach", CODE_COMMIT],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True
).strip()
if actual_commit != CODE_COMMIT:
    raise RuntimeError(
        f"Unexpected code revision: {actual_commit}; expected {CODE_COMMIT}."
    )

required_code = [
    "train_all_folds.py",
    "train_deep_rbf.py",
    "train_deep_rbf_all_folds.py",
    "tune_deep_rbf_thresholds_all_folds.py",
    "export_deep_rbf_bundle.py",
    "model/deep_rbf_system.py",
]
missing_code = [name for name in required_code if not (PROJECT_DIR / name).is_file()]
if missing_code:
    raise FileNotFoundError(f"GitHub code snapshot is incomplete: {missing_code}")

# Compatibility cleanup for the current DeepRBF-only commit:
# reject_distances() no longer accepts the obsolete gap_scope keyword, while
# two utility call sites and the old test still pass it. Remove only that
# obsolete keyword and the test that asserts the retired legacy behavior.
def remove_obsolete_gap_scope(path: Path) -> None:
    source = path.read_text(encoding="utf-8")
    updated = source.replace(', gap_scope="both_inside"', "")
    updated = updated.replace(", gap_scope='both_inside'", "")
    updated = re.sub(
        r"^[ \t]*gap_scope\s*=\s*[\"']both_inside[\"'],?[ \t]*\n",
        "",
        updated,
        flags=re.M,
    )
    if path.name == "test_deep_rbf_thesis.py":
        updated, _ = re.subn(
            r"(?:\A|\n+)def test_legacy_centre_rbf_gap_rule_remains_global_by_default\(\):"
            r"[\s\S]*\Z",
            "",
            updated,
        )
        updated = updated.rstrip() + "\n"
    path.write_text(updated, encoding="utf-8")


for relative in (
    "utils/rbf_pipeline.py",
    "utils/rbf_thresholds.py",
    "tests/test_deep_rbf_thesis.py",
):
    source_path = PROJECT_DIR / relative
    if source_path.is_file():
        remove_obsolete_gap_scope(source_path)

for relative in ("utils/rbf_pipeline.py", "utils/rbf_thresholds.py", "tests/test_deep_rbf_thesis.py"):
    if "gap_scope=" in (PROJECT_DIR / relative).read_text(encoding="utf-8"):
        raise RuntimeError(f"Obsolete gap_scope call remains in {relative}")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

# Keep Colab's preinstalled CUDA-compatible torch/torchvision. Install only
# the other project dependencies so pip does not replace the GPU PyTorch build.
requirements = [
    line.strip()
    for line in (PROJECT_DIR / "requirements.txt").read_text().splitlines()
    if line.strip() and not line.lstrip().startswith("#")
    and not line.lower().startswith(("torch", "torchvision"))
]
if requirements:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *requirements],
        check=True,
    )

import torchvision
import model.rbf as rbf_module
from model.rbf import DeepRBF, reject_distances

if not hasattr(DeepRBF, "forward") or hasattr(rbf_module, "CentreRBF"):
    raise RuntimeError("The active source is not the expected DeepRBF-only implementation.")
# Fail fast if the updated rejection API cannot be imported/executed.
reject_distances(
    torch.tensor([[0.2, 0.8]], device="cuda"),
    distance_threshold=1.0,
    gap_threshold=0.05,
)

DEVICE = "cuda"
TORCH_DEVICE = torch.device("cuda")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Colab GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Git commit:", actual_commit)
print("Dataset version:", dataset_metadata.get("dataset_version"))
print("Dataset directory:", DATASET_DIR)
print("Fold manifest:", MANIFEST)
print("Persistent Drive runs:", RUNS_DIR)
print("Ready: training will execute on the Colab GPU.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive: False
Dataset ZIP: False
Runs: False


AssertionError: Drive folder not found

In [ ]:
patcher_source = '#!/usr/bin/env python3\n"""Patch train.py to write compact, non-resumable checkpoints after final epochs.\n\nIntermediate checkpoints remain fully resumable. Only a checkpoint written at\nits configured final epoch drops optimizer/RNG state. Model weights and all\ninference/provenance metadata are preserved exactly.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nfrom pathlib import Path\n\nOLD_RESUME_GUARD = \'\'\'        if "optimizer_state_dict" not in resume_checkpoint:\n            raise SystemExit("This checkpoint has no optimizer state and cannot be resumed.")\n\'\'\'\nNEW_RESUME_GUARD = \'\'\'        has_optimizer_state = "optimizer_state_dict" in resume_checkpoint\n        compact_complete = (\n            resume_checkpoint.get("checkpoint_storage") == "compact_completed"\n            and resume_checkpoint.get("training_complete") is True\n            and int(resume_checkpoint.get("epoch", -1)) == args.epochs\n        )\n        if not has_optimizer_state and not compact_complete:\n            raise SystemExit(\n                "This checkpoint has no optimizer state. Only a compact checkpoint "\n                "from a fully completed run with the same epoch budget can be reused. "\n                "An incomplete compact checkpoint cannot be resumed."\n            )\n\'\'\'\n\nOLD_OPTIMIZER_LOAD = \'\'\'        optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])\n\'\'\'\nNEW_OPTIMIZER_LOAD = \'\'\'        if "optimizer_state_dict" in resume_checkpoint:\n            optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])\n\'\'\'\n\nOLD_FINAL_SAVE = \'\'\'                _save_checkpoint(last_path, checkpoint_payload(epoch, val_stats))\n\'\'\'\nNEW_FINAL_SAVE = \'\'\'                payload = checkpoint_payload(epoch, val_stats)\n                if epoch == args.epochs:\n                    # A completed fold is used for frozen feature extraction and\n                    # inference. Its optimizer and RNG states are no longer needed.\n                    # Keep every model tensor and all provenance/training metadata.\n                    for key in (\n                        "optimizer_state_dict",\n                        "train_loader_generator_state",\n                        "torch_rng_state",\n                        "cuda_rng_state_all",\n                    ):\n                        payload.pop(key, None)\n                    payload["checkpoint_storage"] = "compact_completed"\n                    payload["training_complete"] = True\n                else:\n                    payload["checkpoint_storage"] = "resumable_full"\n                    payload["training_complete"] = False\n                _save_checkpoint(last_path, payload)\n\'\'\'\n\n\ndef _replace_once(source: str, old: str, new: str, label: str) -> str:\n    # Check the complete replacement first: an old indented line can appear as\n    # a substring inside a newly inserted, more-indented block.\n    if new in source:\n        return source  # already patched\n    count = source.count(old)\n    if count == 1:\n        return source.replace(old, new, 1)\n    raise RuntimeError(\n        f"Could not safely patch {label}: expected one original block or an already-patched block; "\n        f"found {count} original matches. No file was changed."\n    )\n\n\ndef patch_train_file(train_file: Path) -> bool:\n    train_file = Path(train_file)\n    if not train_file.is_file():\n        raise FileNotFoundError(f"train.py not found: {train_file}")\n\n    original = train_file.read_text(encoding="utf-8")\n    updated = original\n    updated = _replace_once(updated, OLD_RESUME_GUARD, NEW_RESUME_GUARD, "compact resume guard")\n    updated = _replace_once(updated, OLD_OPTIMIZER_LOAD, NEW_OPTIMIZER_LOAD, "optimizer restore")\n    updated = _replace_once(updated, OLD_FINAL_SAVE, NEW_FINAL_SAVE, "final checkpoint save")\n\n    if updated == original:\n        print("Compact checkpoint support is already installed in train.py.")\n        return False\n\n    ast.parse(updated, filename=str(train_file))\n    backup = train_file.with_name("train.py.pre_compact_backup")\n    if not backup.exists():\n        backup.write_text(original, encoding="utf-8")\n\n    temporary = train_file.with_name("train.py.compact.tmp")\n    temporary.write_text(updated, encoding="utf-8")\n    ast.parse(temporary.read_text(encoding="utf-8"), filename=str(temporary))\n    temporary.replace(train_file)\n\n    # Confirm all patch markers and ensure the new code remains syntactically valid.\n    final_source = train_file.read_text(encoding="utf-8")\n    ast.parse(final_source, filename=str(train_file))\n    required_markers = (\n        \'"checkpoint_storage"] = "compact_completed"\',\n        \'"training_complete"] = True\',\n        \'if "optimizer_state_dict" in resume_checkpoint:\',\n    )\n    missing = [marker for marker in required_markers if marker not in final_source]\n    if missing:\n        raise RuntimeError(f"Patch verification failed; missing markers: {missing}")\n\n    print(f"Patched: {train_file}")\n    print(f"Backup:  {backup}")\n    print("Intermediate checkpoints: full/resumable")\n    print("Final completed checkpoints: compact (optimizer and RNG states removed)")\n    print("Model weights, loss metadata, preprocessing, fold identity, manifest hash and history are preserved.")\n    return True\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--repo-root", type=Path, required=True)\n    args = parser.parse_args()\n    patch_train_file(args.repo_root / "train.py")\n\n\nif __name__ == "__main__":\n    main()\n'
compactor_source = '#!/usr/bin/env python3\n"""Safely remove optimizer/RNG tensors from a completed Focal WideResNet checkpoint.\n\nThe model state_dict and all provenance/training metadata are preserved. The\nscript refuses to alter a baseline if a DeepRBF checkpoint already references\nits SHA-256, because changing the baseline file would invalidate that link.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport os\nimport shutil\nfrom pathlib import Path\n\nimport torch\n\nDROP_KEYS = {\n    "optimizer_state_dict",\n    "train_loader_generator_state",\n    "torch_rng_state",\n    "cuda_rng_state_all",\n}\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef tensor_bytes(value) -> int:\n    if isinstance(value, torch.Tensor):\n        return value.numel() * value.element_size()\n    if isinstance(value, dict):\n        return sum(tensor_bytes(item) for item in value.values())\n    if isinstance(value, (tuple, list)):\n        return sum(tensor_bytes(item) for item in value)\n    return 0\n\n\ndef _same_state_dict(left: dict, right: dict) -> bool:\n    if left.keys() != right.keys():\n        return False\n    for key in left:\n        a, b = left[key], right[key]\n        if isinstance(a, torch.Tensor):\n            if not isinstance(b, torch.Tensor) or not torch.equal(a, b):\n                return False\n        elif a != b:\n            return False\n    return True\n\n\ndef compact_checkpoint(\n    checkpoint_path: Path,\n    *,\n    fold_id: int,\n    expected_epochs: int,\n    deep_rbf_root: Path | None = None,\n) -> bool:\n    checkpoint_path = Path(checkpoint_path)\n    if not checkpoint_path.is_file():\n        raise FileNotFoundError(checkpoint_path)\n\n    original_hash = sha256_file(checkpoint_path)\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if not isinstance(checkpoint, dict):\n        raise ValueError("Expected a dictionary checkpoint")\n    if checkpoint.get("checkpoint_kind") != "last":\n        raise ValueError("Only last.pt checkpoints can be compacted")\n    if checkpoint.get("architecture") != "wide_resnet50_2":\n        raise ValueError("Expected a WideResNet50-2 checkpoint")\n    if checkpoint.get("label_source") != "consensus":\n        raise ValueError("Expected the consensus-label baseline")\n    if checkpoint.get("loss", {}).get("name") != "focal":\n        raise ValueError("Expected a Focal Loss baseline")\n    if int(checkpoint.get("fold_id", -1)) != fold_id:\n        raise ValueError("Checkpoint fold_id does not match --fold-id")\n    if int(checkpoint.get("epoch", -1)) != expected_epochs:\n        raise ValueError(\n            f"Checkpoint is not complete: epoch={checkpoint.get(\'epoch\')}, "\n            f"expected={expected_epochs}"\n        )\n    if "model_state_dict" not in checkpoint:\n        raise ValueError("Checkpoint is missing model_state_dict")\n\n    if checkpoint.get("checkpoint_storage") == "compact_completed":\n        print(f"Already compact: {checkpoint_path}")\n        return False\n\n    if deep_rbf_root is not None:\n        # A DeepRBF stage checkpoint stores the exact baseline file hash.\n        # Never change a baseline that has already been used to train DeepRBF.\n        candidates = list(Path(deep_rbf_root).glob(f"fold{fold_id:02d}_*/last.pt"))\n        if candidates:\n            raise RuntimeError(\n                f"Refusing to compact {checkpoint_path}: a DeepRBF checkpoint already exists "\n                f"for fold {fold_id} ({candidates[0]}). The DeepRBF source-baseline SHA-256 "\n                "may depend on the current checkpoint."\n            )\n\n    if "optimizer_state_dict" not in checkpoint:\n        raise ValueError("Checkpoint is neither a full resumable checkpoint nor a recognized compact checkpoint")\n\n    original_model_state = checkpoint["model_state_dict"]\n    original_model_bytes = tensor_bytes(original_model_state)\n    compact = {key: value for key, value in checkpoint.items() if key not in DROP_KEYS}\n    compact["checkpoint_storage"] = "compact_completed"\n    compact["training_complete"] = True\n    compact["compact_removed_keys"] = sorted(DROP_KEYS.intersection(checkpoint))\n    compact["source_full_checkpoint_sha256"] = original_hash\n\n    needed = int(tensor_bytes(compact) * 1.05) + 32 * 1024**2\n    free = shutil.disk_usage(checkpoint_path.parent).free\n    if free < needed:\n        raise RuntimeError(\n            f"Not enough temporary space to compact safely: need {needed / 1e9:.2f} GB, "\n            f"free {free / 1e9:.2f} GB. Original checkpoint was not changed."\n        )\n\n    temporary = checkpoint_path.with_name(checkpoint_path.name + ".compact.tmp")\n    try:\n        torch.save(compact, temporary)\n        verified = torch.load(temporary, map_location="cpu", weights_only=True)\n        if not _same_state_dict(original_model_state, verified.get("model_state_dict", {})):\n            raise RuntimeError("Model weights changed during compact checkpoint serialization")\n        if any(key in verified for key in DROP_KEYS):\n            raise RuntimeError("Optimizer/RNG state remains in the compact checkpoint")\n        for key in (\n            "architecture", "fold_id", "epoch", "loss", "training_config",\n            "fold_manifest_sha256", "preprocessing", "class_names", "num_classes",\n        ):\n            if verified.get(key) != checkpoint.get(key):\n                raise RuntimeError(f"Checkpoint metadata changed during compaction: {key}")\n        if verified.get("checkpoint_storage") != "compact_completed":\n            raise RuntimeError("Compact checkpoint marker is missing")\n        if sha256_file(checkpoint_path) != original_hash:\n            raise RuntimeError("Original checkpoint changed before atomic replacement")\n        os.replace(temporary, checkpoint_path)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n\n    final = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if not _same_state_dict(original_model_state, final["model_state_dict"]):\n        raise RuntimeError("Post-replacement model-weight verification failed")\n\n    new_size = checkpoint_path.stat().st_size\n    print(f"Compacted: {checkpoint_path}")\n    print(f"Model weights preserved: {original_model_bytes / 1024**2:.2f} MB")\n    print(f"New checkpoint size: {new_size / 1024**2:.2f} MB")\n    print(f"Original SHA-256: {original_hash}")\n    print(f"Compact SHA-256:  {sha256_file(checkpoint_path)}")\n    return True\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--checkpoint", type=Path, required=True)\n    parser.add_argument("--fold-id", type=int, required=True)\n    parser.add_argument("--expected-epochs", type=int, default=15)\n    parser.add_argument("--deep-rbf-root", type=Path, default=None)\n    args = parser.parse_args()\n    compact_checkpoint(\n        args.checkpoint,\n        fold_id=args.fold_id,\n        expected_epochs=args.expected_epochs,\n        deep_rbf_root=args.deep_rbf_root,\n    )\n\n\nif __name__ == "__main__":\n    main()\n'

patcher_path = PROJECT_DIR / "apply_compact_checkpoint_patch.py"
compactor_path = PROJECT_DIR / "compact_wideresnet_checkpoint.py"
patcher_path.write_text(patcher_source, encoding="utf-8")
compactor_path.write_text(compactor_source, encoding="utf-8")
subprocess.run(
    [sys.executable, str(patcher_path), "--repo-root", str(PROJECT_DIR)],
    cwd=PROJECT_DIR,
    check=True,
)
print("Compact completed-checkpoint support is active.")


## 0A. Preflight — verify the DeepRBF implementation before spending GPU time

This runs the focused DeepRBF architecture, loss, rejection-rule, and combined-bundle tests. Training cells should only be run if this cell passes.

The setup also installs a compact-checkpoint patch in `train.py`. Intermediate checkpoints remain full and resumable; at the final epoch, the exact model weights and provenance metadata are retained while optimizer and RNG states are removed. The already-trained Fold 1 checkpoint is not rewritten because its existing DeepRBF checkpoint is bound to its original SHA-256.


In [ ]:
smoke_test_command = [
    sys.executable, "-m", "pytest", "-q",
    "tests/test_deep_rbf_thesis.py",
    "tests/test_deep_rbf_system.py",
]
subprocess.run(smoke_test_command, cwd=PROJECT_DIR, check=True)
print("DeepRBF preflight tests passed. GPU:", torch.cuda.get_device_name(0))

## 1. Configuration — start with one fold

The baseline output directory matches the team's existing focal-baseline notebook, so a completed matching Focal checkpoint can be resumed/reused rather than trained again. The DeepRBF outputs are kept in a separate Drive folder.

Enter the approximate free Google Drive storage in `DRIVE_FREE_GB`. The conservative estimate below includes one baseline checkpoint and one combined bundle per selected fold, plus a safety margin. For the one-fold pilot, allow at least about 3.1 GB; for all 25 folds, allow about 29.5 GB.

Do not change the training recipe halfway through a run. If you change the recipe, use a new output folder.

For a 15 GB Drive, keep `DELETE_STAGE_CHECKPOINTS_AFTER_BUNDLE = True`. After a combined bundle is exported, loaded successfully and validation-evaluated, the duplicate WideResNet and DeepRBF source `.pt` files are removed; the combined model, histories, threshold configs and validation reports remain. Set `DRIVE_FREE_GB` to the *current free space*, not the total account quota.


In [ ]:
# Full development run. Fold 1 already exists and will be reused; folds 2-25
# will be trained and saved as compact completed checkpoints.
FOLD_START = 1
FOLD_END = 25

SEED = 42

# Stage 1: WideResNet50-2 + Focal Loss (same settings as the team's baseline notebook)
BASELINE_EPOCHS = 15
BASELINE_BATCH_SIZE = 8
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA = 2.0
FOCAL_ALPHA_MODE = "uniform"
AUGMENTATION = "standard"
SAMPLING = "class_balanced"
ROTATION_DEGREES = 10.0
HFLIP_PROB = 0.5
NOISE_STD = 0.01
CHECKPOINT_EVERY = 5

# Stage 2: DeepRBF
RBF_EPOCHS = 30
RBF_DIM = 64
RBF_DISTANCE = "l1"       # L1 and L2 need separate experiments/thresholds
RBF_DISTANCE_POWER = "norm"
KNOWN_MARGIN = 1.0
REJECT_MARGIN = 1.0
BETA = 1.0
RBF_LR = 1e-3
RBF_WEIGHT_DECAY = 0.0
FEATURE_BATCH_SIZE = 8

# After each combined bundle has been exported, loaded successfully and
# evaluated on validation, duplicate source .pt files are removed. The bundle
# retains both sets of exact model weights; histories/reports remain separately.
DELETE_STAGE_CHECKPOINTS_AFTER_BUNDLE = True

# Enter the ACTUAL free space currently available in Google Drive, not the
# account's total quota. Example: enter 11.2 if Drive currently shows 11.2 GB free.
DRIVE_FREE_GB = None

BASELINE_RUNS = DRIVE_DIR / "runs" / "baselines_shared_dataset_v1"
BASELINE_DIR = BASELINE_RUNS / "focal" / "checkpoints"
RBF_ROOT = DRIVE_DIR / "runs" / "deeprbf_focal_shared_dataset_v1"

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
RBF_ROOT.mkdir(parents=True, exist_ok=True)

selected_fold_ids = list(range(FOLD_START, FOLD_END + 1))
selected_folds = len(selected_fold_ids)
if DRIVE_FREE_GB is None:
    raise RuntimeError(
        "Set DRIVE_FREE_GB to the actual free space shown in Google Drive before training."
    )


def _tensor_bytes(value):
    if isinstance(value, torch.Tensor):
        return value.numel() * value.element_size()
    if isinstance(value, dict):
        return sum(_tensor_bytes(item) for item in value.values())
    if isinstance(value, (tuple, list)):
        return sum(_tensor_bytes(item) for item in value)
    return 0


# Files already present consume space that is reflected in DRIVE_FREE_GB.
# Estimate only the additional space needed for missing folds.
existing_baselines = {}
for path in BASELINE_DIR.glob("*_last.pt"):
    match = re.search(r"_fold(\d{2})_", path.name)
    if match:
        fold_id = int(match.group(1))
        if fold_id in selected_fold_ids:
            if fold_id in existing_baselines:
                raise RuntimeError(f"Multiple WideResNet checkpoints found for fold {fold_id}.")
            existing_baselines[fold_id] = path

reference_baseline = next(iter(BASELINE_DIR.glob("*_last.pt")), None)
if reference_baseline is None:
    raise RuntimeError(
        "No existing WideResNet checkpoint is available to measure model size. "
        "Run the one-fold pilot first, then use this compact 25-fold notebook."
    )
reference_payload = torch.load(reference_baseline, map_location="cpu", weights_only=True)
model_state_bytes = _tensor_bytes(reference_payload["model_state_dict"])
del reference_payload

existing_rbf = {}
for path in RBF_ROOT.glob(f"fold*_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}/last.pt"):
    match = re.match(r"fold(\d{2})_", path.parent.name)
    if match:
        fold_id = int(match.group(1))
        if fold_id in selected_fold_ids:
            existing_rbf[fold_id] = path

if existing_rbf:
    sample_rbf = torch.load(next(iter(existing_rbf.values())), map_location="cpu", weights_only=True)
    rbf_state_bytes = _tensor_bytes(sample_rbf.get("model_state_dict", {}))
    del sample_rbf
else:
    rbf_state_bytes = 5 * 1024**2

missing_baseline_folds = [f for f in selected_fold_ids if f not in existing_baselines]
missing_rbf_folds = [f for f in selected_fold_ids if f not in existing_rbf]
additional_baseline_bytes = len(missing_baseline_folds) * model_state_bytes
additional_rbf_bytes = len(missing_rbf_folds) * max(rbf_state_bytes, 5 * 1024**2)

# A full checkpoint can coexist with its temporary atomic-save file during
# intermediate epochs. One bundle is staged at a time if source cleanup is on.
full_checkpoint_temp_reserve = 800_000_000
bundle_bytes = model_state_bytes + max(rbf_state_bytes, 5 * 1024**2) + 5 * 1024**2
bundle_reserve = bundle_bytes if DELETE_STAGE_CHECKPOINTS_AFTER_BUNDLE else bundle_bytes * selected_folds
safety_margin = 1_000_000_000
estimated_required_bytes = (
    additional_baseline_bytes
    + additional_rbf_bytes
    + full_checkpoint_temp_reserve
    + bundle_reserve
    + safety_margin
)
estimated_required_gb = estimated_required_bytes / 1e9

if DRIVE_FREE_GB < estimated_required_gb:
    raise RuntimeError(
        f"Insufficient Drive space: {DRIVE_FREE_GB:.2f} GB free; "
        f"the compact pipeline estimates about {estimated_required_gb:.2f} GB additional "
        "space, including temporary saves, bundle staging and a safety margin. "
        "Do not start the 25-fold run until more space is available."
    )

print("Folds:", FOLD_START, "to", FOLD_END)
print("GPU:", torch.cuda.get_device_name(0))
print("Existing WideResNet folds:", sorted(existing_baselines))
print("Existing DeepRBF folds:", sorted(existing_rbf))
print(f"Estimated additional compact baseline storage: {additional_baseline_bytes / 1e9:.2f} GB")
print(f"Estimated additional DeepRBF storage: {additional_rbf_bytes / 1e9:.2f} GB")
print(f"Combined bundle peak reserve: {bundle_reserve / 1e9:.2f} GB")
print(f"Temporary-save + safety reserve: {(full_checkpoint_temp_reserve + safety_margin) / 1e9:.2f} GB")
print(f"Estimated additional Drive requirement: {estimated_required_gb:.2f} GB")
print("Focal checkpoints:", BASELINE_DIR)
print("DeepRBF outputs:", RBF_ROOT)


## 2. Stage 1 — train WideResNet50-2 with Focal Loss

This uses the same `train_all_folds.py` training script, fold manifest, augmentation settings, optimizer settings, checkpoint cadence, and Drive checkpoint layout as the teammate's `01_wideresnet50_2.ipynb`.

Only the **Focal Loss** experiment is run here; the CE comparison is not needed for the requested DeepRBF pipeline. `--resume` reuses completed matching folds and resumes incomplete ones.

**Storage policy:** intermediate epoch checkpoints remain full so interrupted training can resume. At the final epoch, the checkpoint keeps the full WideResNet weights, Focal metadata, fold/manifest/preprocessing identity and history, but drops the optimizer and RNG states. Fold 1 is reused unchanged because its existing DeepRBF checkpoint is cryptographically bound to that original baseline file.


In [ ]:
from utils.notebook_training import run_with_progress

for fold in range(FOLD_START, FOLD_END + 1):
    fold_name = f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
    bundle_path = RBF_ROOT / fold_name / "combined_model.pt"
    if bundle_path.is_file():
        print(f"Fold {fold:02d}: verified combined bundle already exists; skip Stage 1.")
        continue

    baseline_command = [
        sys.executable, "-u", "train_all_folds.py",
        "--fold-start", str(fold), "--fold-end", str(fold),
        "--loss", "focal", "--focal-gamma", str(FOCAL_GAMMA),
        "--focal-alpha-mode", FOCAL_ALPHA_MODE,
        "--augmentation", AUGMENTATION,
        "--rotation-degrees", str(ROTATION_DEGREES),
        "--hflip-prob", str(HFLIP_PROB), "--noise-std", str(NOISE_STD),
        "--sampling", SAMPLING, "--label-source", "consensus",
        "--pretrained", "--freeze-policy", "early",
        "--epochs", str(BASELINE_EPOCHS), "--batch-size", str(BASELINE_BATCH_SIZE),
        "--image-size", "224", "--num-workers", "0",
        "--lr", str(HEAD_LR), "--backbone-lr", str(BACKBONE_LR),
        "--weight-decay", str(WEIGHT_DECAY), "--seed", str(SEED),
        "--device", DEVICE, "--manifest", str(MANIFEST),
        "--root", str(DATA_ROOT), "--checkpoint-dir", str(BASELINE_DIR),
        "--checkpoint-every", str(CHECKPOINT_EVERY), "--progress-json", "--resume",
    ]
    print("\nStarting/reusing Focal WideResNet fold", fold)
    print(" ".join(baseline_command))
    run_with_progress(
        baseline_command, title="WideResNet50-2 + Focal Loss",
        fold_start=fold, fold_end=fold, epochs=BASELINE_EPOCHS,
    )

    matches = [
        p for p in BASELINE_DIR.glob("*_last.pt")
        if re.search(rf"_fold{fold:02d}_", p.name)
    ]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one final baseline checkpoint for fold {fold}; found {matches}")

    rbf_checkpoint_path = RBF_ROOT / f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}" / "last.pt"
    if fold != 1 and not rbf_checkpoint_path.is_file():
        compact_command = [
            sys.executable, "compact_wideresnet_checkpoint.py",
            "--checkpoint", str(matches[0]),
            "--fold-id", str(fold),
            "--expected-epochs", str(BASELINE_EPOCHS),
            "--deep-rbf-root", str(RBF_ROOT),
        ]
        subprocess.run(compact_command, cwd=PROJECT_DIR, check=True)
    elif rbf_checkpoint_path.is_file():
        print(f"Fold {fold:02d}: preserving baseline file because a DeepRBF checkpoint already exists.")

    saved = torch.load(matches[0], map_location="cpu", weights_only=True)
    if int(saved.get("epoch", -1)) != BASELINE_EPOCHS:
        raise RuntimeError(f"Fold {fold} did not reach {BASELINE_EPOCHS} epochs.")
    if fold != 1 and not rbf_checkpoint_path.is_file() and saved.get("checkpoint_storage") != "compact_completed":
        raise RuntimeError(
            f"Fold {fold} is not compact. Refusing to continue; check the compact checkpoint patch."
        )
    print(
        f"Fold {fold:02d} complete | epoch={saved['epoch']} | "
        f"file_size={matches[0].stat().st_size / 1024**2:.1f} MB | "
        f"storage={saved.get('checkpoint_storage', 'legacy_full')}"
    )
    del saved

print("Stage 1 complete for selected folds.")


## 3. Stage 2 — train DeepRBF on the frozen Focal-trained features

The trainer finds the matching Focal checkpoint for every requested fold and verifies the fold ID, class mapping, 2048-D feature width, and exact manifest hash.

The WideResNet is frozen and used only to extract features. DeepRBF is trained independently; Focal Loss is not applied a second time. `--resume` continues from the last saved DeepRBF checkpoint after a Colab runtime reset.

In [ ]:
for fold in range(FOLD_START, FOLD_END + 1):
    fold_name = f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
    fold_dir = RBF_ROOT / fold_name
    bundle_path = fold_dir / "combined_model.pt"
    if bundle_path.is_file():
        print(f"Fold {fold:02d}: combined bundle already exists; skip Stage 2.")
        continue

    baseline_matches = [
        p for p in BASELINE_DIR.glob("*_last.pt")
        if re.search(rf"_fold{fold:02d}_", p.name)
    ]
    if len(baseline_matches) != 1:
        raise RuntimeError(
            f"Fold {fold:02d}: expected one matching Focal baseline before DeepRBF training; "
            f"found {baseline_matches}."
        )

    rbf_command = [
        sys.executable, "-u", "train_deep_rbf_all_folds.py",
        "--baseline-dir", str(BASELINE_DIR), "--baseline-glob", "*_last.pt",
        "--manifest", str(MANIFEST), "--root", str(DATA_ROOT),
        "--output-root", str(RBF_ROOT), "--fold-start", str(fold), "--fold-end", str(fold),
        "--epochs", str(RBF_EPOCHS), "--rbf-dim", str(RBF_DIM),
        "--distance", RBF_DISTANCE, "--distance-power", RBF_DISTANCE_POWER,
        "--known-margin", str(KNOWN_MARGIN), "--reject-margin", str(REJECT_MARGIN),
        "--beta", str(BETA), "--balance-groups", "--lr", str(RBF_LR),
        "--weight-decay", str(RBF_WEIGHT_DECAY), "--seed", str(SEED),
        "--device", DEVICE, "--feature-batch-size", str(FEATURE_BATCH_SIZE), "--resume",
    ]
    print("\nTraining/reusing DeepRBF fold", fold)
    print(" ".join(rbf_command))
    subprocess.run(rbf_command, cwd=PROJECT_DIR, check=True)

print("Stage 2 complete for selected folds.")


## 4. Stage 3A — select rejection thresholds using development validation only

The tuner prefers a feasible threshold selected from each fold's own validation data. Fold 22 has no validation images, so it is skipped for local fitting and receives the pooled out-of-fold development-validation fallback if that fallback is feasible.

This step does not load the locked test partition. A one-fold pilot may not contain enough validation participants for a feasible threshold; if so, the notebook preserves the trained weights and defers bundle export until more development folds are available.

In [ ]:
decision_paths = [
    RBF_ROOT / f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
    / "threshold_selection" / "rbf_decision.json"
    for fold in range(FOLD_START, FOLD_END + 1)
]
threshold_report_path = RBF_ROOT / "threshold_selection_global" / "threshold_search_report.json"

THRESHOLDS_READY = False
if threshold_report_path.is_file() and all(path.is_file() for path in decision_paths):
    existing_report = json.loads(threshold_report_path.read_text(encoding="utf-8"))
    if existing_report.get("status") == "selected":
        THRESHOLDS_READY = True
        print("Reusing the existing successful validation-only threshold selection.")

if not THRESHOLDS_READY:
    threshold_command = [
        sys.executable, "-u", "tune_deep_rbf_thresholds_all_folds.py",
        "--baseline-dir", str(BASELINE_DIR), "--baseline-glob", "*_last.pt",
        "--deep-rbf-root", str(RBF_ROOT), "--manifest", str(MANIFEST),
        "--root", str(DATA_ROOT), "--fold-start", str(FOLD_START),
        "--fold-end", str(FOLD_END), "--distance", RBF_DISTANCE,
        "--distance-power", RBF_DISTANCE_POWER, "--selection-unit", "participants",
        "--max-pd-to-control-rate", "0.0", "--min-coverage", "0.0",
        "--feature-batch-size", str(FEATURE_BATCH_SIZE), "--device", DEVICE,
    ]
    if threshold_report_path.exists():
        threshold_report_path.unlink()
    print(" ".join(threshold_command))
    threshold_run = subprocess.run(threshold_command, cwd=PROJECT_DIR, check=False)
    if not threshold_report_path.is_file():
        raise RuntimeError(
            f"Threshold tuner failed before writing its report (exit={threshold_run.returncode})."
        )
    threshold_report = json.loads(threshold_report_path.read_text(encoding="utf-8"))
    THRESHOLDS_READY = threshold_report.get("status") == "selected"

if not THRESHOLDS_READY:
    print(
        "No feasible threshold was found for this fold selection. The trained WideResNet "
        "and DeepRBF checkpoints are preserved in Drive. Do not invent a threshold; "
        "train additional development folds and rerun this cell."
    )
else:
    print("Threshold selection complete. Fold-specific decision files are ready.")


## 5. Stage 3B — export combined bundles and evaluate validation (when thresholds are feasible)

Each bundle contains:
- the trained Focal WideResNet weights and preprocessing metadata;
- the separately trained DeepRBF weights and configuration;
- the matching validation-selected rejection thresholds;
- provenance hashes tying the stages together.

The combined model uses `WideResNet.forward_features(images) → DeepRBF(features)`. The baseline classifier logits are not averaged with RBF distances.

In [ ]:
if THRESHOLDS_READY:
    from model.deep_rbf_system import load_deep_rbf_bundle

    def find_baseline_checkpoint(fold_id):
        matches = [
            p for p in BASELINE_DIR.glob("*_last.pt")
            if re.search(rf"_fold{fold_id:02d}_", p.name)
        ]
        if len(matches) != 1:
            raise RuntimeError(
                f"Expected exactly one baseline checkpoint for fold {fold_id}; found {matches}"
            )
        return matches[0]

    for fold in range(FOLD_START, FOLD_END + 1):
        fold_name = f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
        fold_dir = RBF_ROOT / fold_name
        rbf_checkpoint = fold_dir / "last.pt"
        decision_config = fold_dir / "threshold_selection" / "rbf_decision.json"
        bundle_path = fold_dir / "combined_model.pt"
        validation_dir = fold_dir / "combined_validation"
        validation_report = validation_dir / "val_report.json"

        if bundle_path.is_file() and validation_report.is_file():
            loaded_system, loaded_bundle = load_deep_rbf_bundle(bundle_path, device=TORCH_DEVICE)
            if int(loaded_bundle.get("fold_id", -1)) != fold:
                raise RuntimeError(f"Existing combined bundle has the wrong fold ID: {bundle_path}")
            del loaded_system, loaded_bundle
            print(f"Fold {fold:02d}: verified existing combined bundle; skip export/evaluation.")
            if rbf_checkpoint.is_file() and not (fold_dir / "training_history.json").is_file():
                old_rbf_payload = torch.load(rbf_checkpoint, map_location="cpu", weights_only=True)
                old_history_record = {
                    "format_version": "deep-rbf-training-history-v1",
                    "fold_id": int(old_rbf_payload.get("fold_id", fold)),
                    "epoch": int(old_rbf_payload.get("epoch", 0)),
                    "epochs_requested": int(old_rbf_payload.get("epochs_requested", RBF_EPOCHS)),
                    "config": old_rbf_payload.get("config", {}),
                    "history": old_rbf_payload.get("history", []),
                    "source_baseline": {
                        "sha256": old_rbf_payload.get("source_baseline", {}).get("sha256"),
                        "fold_manifest_sha256": old_rbf_payload.get("source_baseline", {}).get("fold_manifest_sha256"),
                        "fold_id": old_rbf_payload.get("source_baseline", {}).get("fold_id"),
                        "baseline_epoch": old_rbf_payload.get("source_baseline", {}).get("baseline_epoch"),
                    },
                }
                (fold_dir / "training_history.json").write_text(
                    json.dumps(old_history_record, ensure_ascii=False, indent=2) + "\n",
                    encoding="utf-8",
                )
                del old_rbf_payload
            if DELETE_STAGE_CHECKPOINTS_AFTER_BUNDLE:
                for old_path in [*BASELINE_DIR.glob(f"*_fold{fold:02d}_*_last.pt"), rbf_checkpoint]:
                    if old_path.is_file():
                        old_path.unlink()
            continue

        if not rbf_checkpoint.is_file():
            raise FileNotFoundError(rbf_checkpoint)
        if not decision_config.is_file():
            raise FileNotFoundError(decision_config)
        baseline_path = find_baseline_checkpoint(fold)

        # Preserve DeepRBF history in JSON before removing its optimizer-bearing .pt.
        rbf_payload = torch.load(rbf_checkpoint, map_location="cpu", weights_only=True)
        history_path = fold_dir / "training_history.json"
        history_record = {
            "format_version": "deep-rbf-training-history-v1",
            "fold_id": int(rbf_payload.get("fold_id", fold)),
            "epoch": int(rbf_payload.get("epoch", 0)),
            "epochs_requested": int(rbf_payload.get("epochs_requested", RBF_EPOCHS)),
            "config": rbf_payload.get("config", {}),
            "history": rbf_payload.get("history", []),
            "source_baseline": {
                "sha256": rbf_payload.get("source_baseline", {}).get("sha256"),
                "fold_manifest_sha256": rbf_payload.get("source_baseline", {}).get("fold_manifest_sha256"),
                "fold_id": rbf_payload.get("source_baseline", {}).get("fold_id"),
                "baseline_epoch": rbf_payload.get("source_baseline", {}).get("baseline_epoch"),
            },
        }
        history_path.write_text(
            json.dumps(history_record, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        del rbf_payload

        export_command = [
            sys.executable, "export_deep_rbf_bundle.py",
            "--deep-rbf-checkpoint", str(rbf_checkpoint),
            "--baseline-checkpoint", str(baseline_path),
            "--decision-config", str(decision_config),
            "--manifest", str(MANIFEST), "--root", str(DATA_ROOT),
            "--output", str(bundle_path), "--device", DEVICE,
        ]
        subprocess.run(export_command, cwd=PROJECT_DIR, check=True)

        # Validate on development validation before deleting any source checkpoint.
        validation_command = [
            sys.executable, "eval_deep_rbf.py",
            "--checkpoint", str(rbf_checkpoint),
            "--baseline-checkpoint", str(baseline_path),
            "--manifest", str(MANIFEST), "--root", str(DATA_ROOT),
            "--split", "val", "--decision-config", str(decision_config),
            "--feature-batch-size", str(FEATURE_BATCH_SIZE), "--device", DEVICE,
            "--output-dir", str(validation_dir),
        ]
        subprocess.run(validation_command, cwd=PROJECT_DIR, check=True)
        if not validation_report.is_file():
            raise RuntimeError(f"Validation report was not created: {validation_report}")

        loaded_system, loaded_bundle = load_deep_rbf_bundle(bundle_path, device=TORCH_DEVICE)
        if int(loaded_bundle.get("fold_id", -1)) != fold:
            raise RuntimeError(f"Exported bundle has the wrong fold ID: {bundle_path}")
        del loaded_system, loaded_bundle

        if DELETE_STAGE_CHECKPOINTS_AFTER_BUNDLE:
            # Combined bundle contains both exact model weights, decision config
            # and provenance hashes. Histories, threshold files and reports remain.
            baseline_path.unlink()
            rbf_checkpoint.unlink()
            print(f"Fold {fold:02d}: verified bundle; removed duplicate source .pt files.")
        else:
            print(f"Fold {fold:02d}: verified bundle; source checkpoints retained.")

    print("Combined bundles and validation reports are saved under:", RBF_ROOT)
else:
    print("Bundle export/evaluation is deferred until a feasible development threshold exists.")


## 6. Compose the two trained models in Python

This cell demonstrates the actual composed forward pass on one nonempty development validation fold. It reports distances and decisions; it does not use expert or clinical labels as model inputs.

For a full 25-fold run, fold 1 is selected for this demonstration. If you run only fold 22, choose another fold with nonempty validation.

In [ ]:
from model.deep_rbf_system import load_deep_rbf_bundle, WideResNetDeepRBF
from utils.rbf_pipeline import load_deep_rbf_checkpoint
from config import PreprocessingConfig
from data.dataloader import SpiralDataset, build_eval_transform
from torch.utils.data import DataLoader

demo_fold = next((f for f in range(FOLD_START, FOLD_END + 1) if f != 22), None)
if demo_fold is None:
    raise ValueError("The selected fold range contains no nonempty validation fold.")

demo_dir = RBF_ROOT / f"fold{demo_fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
if THRESHOLDS_READY:
    system, bundle = load_deep_rbf_bundle(
        demo_dir / "combined_model.pt", device=TORCH_DEVICE
    )
    decision = bundle["decision"]
    preprocessing_data = bundle["baseline"]["preprocessing"]
else:
    # Forward-only composition for a pilot whose validation set is too small
    # to select a defensible rejection threshold.
    backbone, rbf_head, rbf_checkpoint = load_deep_rbf_checkpoint(
        demo_dir / "last.pt", MANIFEST, DATA_ROOT, TORCH_DEVICE
    )
    system = WideResNetDeepRBF(backbone, rbf_head).to(TORCH_DEVICE).eval()
    preprocessing_data = rbf_checkpoint["source_baseline"]["preprocessing"]
    decision = None

preprocessing = PreprocessingConfig.from_dict(preprocessing_data["config"])
val_dataset = SpiralDataset(
    manifest_path=MANIFEST,
    root=DATA_ROOT,
    fold_id=demo_fold,
    split="val",
    image_size=preprocessing.image_size,
    label_source="consensus",
    transform=build_eval_transform(preprocessing),
    allow_empty=True,
)
if len(val_dataset) == 0:
    print(f"Fold {demo_fold} has empty validation; select another fold for the demo.")
else:
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)
    batch = next(iter(val_loader))
    with torch.no_grad():
        distances = system(batch["image"].to(TORCH_DEVICE))
    if decision is not None:
        labels, reasons, distances = system.predict(
            batch["image"].to(TORCH_DEVICE),
            distance_threshold=float(decision["distance_threshold"]),
            gap_threshold=float(decision["gap_threshold"]),
        )
        names = {0: "control", 1: "parkinson", 2: "unknown"}
        for i, drawing_id in enumerate(batch["drawing_id"]):
            print({
                "drawing_id": drawing_id,
                "distance_control": float(distances[i, 0].cpu()),
                "distance_parkinson": float(distances[i, 1].cpu()),
                "prediction": names[int(labels[i].cpu())],
                "reason": reasons[i],
            })
    else:
        print("Combined forward pass succeeded. No threshold was selected on this small pilot.")
        for i, drawing_id in enumerate(batch["drawing_id"]):
            print({
                "drawing_id": drawing_id,
                "distance_control": float(distances[i, 0].cpu()),
                "distance_parkinson": float(distances[i, 1].cpu()),
            })

## 7. Final checks before any held-out test evaluation

- Confirm that every requested fold has a `combined_model.pt` and a `combined_validation/val_report.json`; Fold 22 is expected to report `empty_validation`.
- Confirm that the global threshold report lists the expected nonempty validation folds; fold 22 should be skipped for threshold fitting.
- Review the selected radius/gap and participant-level validation reports.
- Training histories, baseline validation reports, threshold configs and DeepRBF JSON histories remain in Drive.
- With source-checkpoint cleanup enabled, the combined bundles are the retained inference artifacts; the original optimizer-bearing stage `.pt` files have been removed only after bundle verification and validation evaluation.
- Freeze the full protocol before opening the held-out test partition.

The notebook deliberately stops at development validation. Do not select hyperparameters or thresholds from the test partition.